# 06_gold_star_schema
Builds the de-identified star schema in `health.gold` from both sources.

| Table | Grain |
| --- | --- |
| `dim_patient` | one row per patient: pseudonymous key, birth year, sex, 3-digit ZIP |
| `dim_facility` | one row per hospital or clinic |
| `dim_diagnosis` | one row per SNOMED CT code |
| `dim_date` | one row per calendar day |
| `fact_encounter` | one row per encounter, both sources combined |
| `fact_encounter_diagnosis` | bridge: one row per encounter and diagnosis |

**De-identification:** patient and encounter IDs are replaced by salted SHA-256 hashes; names, full birth dates and full ZIP codes do not reach gold.
Encounter dates are kept because length of stay and readmission depend on them, so gold is pseudonymised rather than fully anonymous.
In production the salt would come from a secret scope, and dates could be shifted per patient.

In [0]:
%sql
SET TIME ZONE 'UTC';

-- The salt makes the hashes impossible to reverse by hashing known IDs. Production: read it from a secret scope.
DECLARE OR REPLACE VARIABLE key_salt STRING DEFAULT 'healthcare-lakehouse-demo-salt';

## dim_patient

In [0]:
%sql
CREATE OR REPLACE TABLE health.gold.dim_patient
COMMENT 'One row per patient. Pseudonymous key; birth year, sex, state and 3-digit ZIP only.'
AS
SELECT
  sha2(concat(key_salt, '|patient|', f.patient_id), 256)  AS patient_key,
  year(f.birth_date)                                       AS birth_year,
  f.sex,
  f.state,
  left(f.zip, 3)                                           AS zip3,
  f.deceased_ts IS NOT NULL                                AS is_deceased,
  year(f.deceased_ts)                                      AS deceased_year,
  x.fhir_patient_id IS NOT NULL                            AS in_hl7_feed
FROM health.silver.fhir_patients f
LEFT JOIN (SELECT DISTINCT fhir_patient_id FROM health.silver.patient_xref) x
  ON x.fhir_patient_id = f.patient_id;

## dim_facility (with an Unknown member so no fact row is ever orphaned)

In [0]:
%sql
CREATE OR REPLACE TABLE health.gold.dim_facility
COMMENT 'One row per hospital or clinic, plus an UNKNOWN member.'
AS
SELECT organization_id AS facility_key, organization_name AS facility_name, organization_type, city, state, zip
FROM health.silver.fhir_organizations
UNION ALL
SELECT 'UNKNOWN', 'Unknown facility', NULL, NULL, NULL, NULL;

## dim_diagnosis

In [0]:
%sql
CREATE OR REPLACE TABLE health.gold.dim_diagnosis
COMMENT 'One row per SNOMED CT diagnosis code seen in either source.'
AS
SELECT code AS diagnosis_code, max(display) AS diagnosis_name, 'SNOMED CT' AS code_system
FROM (
  SELECT code, display FROM health.silver.fhir_conditions
  UNION ALL
  SELECT d.code, d.display FROM health.silver.hl7_encounters LATERAL VIEW explode(diagnoses) t AS d
)
WHERE code IS NOT NULL AND code <> ''
GROUP BY code;

## dim_date (every day from the first encounter to the last)

In [0]:
%sql
CREATE OR REPLACE TABLE health.gold.dim_date
COMMENT 'Calendar dimension, one row per day.'
AS
SELECT
  CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
  d                                       AS full_date,
  year(d)                                 AS year,
  quarter(d)                              AS quarter,
  concat(year(d), '-Q', quarter(d))       AS year_quarter,
  month(d)                                AS month,
  date_format(d, 'MMMM')                  AS month_name,
  date_trunc('MONTH', d)::DATE            AS month_start_date,
  dayofweek(d)                            AS day_of_week,
  date_format(d, 'EEEE')                  AS day_name,
  dayofweek(d) IN (1, 7)                  AS is_weekend
FROM (
  SELECT explode(sequence(
           (SELECT to_date(min(start_ts)) FROM health.silver.fhir_encounters),
           (SELECT to_date(max(coalesce(end_ts, start_ts))) FROM health.silver.fhir_encounters),
           INTERVAL 1 DAY)) AS d
);

## fact_encounter
FHIR is the complete record, so every FHIR encounter becomes a fact row, enriched with what only the HL7 feed knows (transfers).
`source_systems` shows whether the visit was also reported by the legacy feed. Any HL7 visit that could not be linked would be added as an HL7-only row.

**30-day readmission:** an inpatient stay is flagged if the same patient is admitted as an inpatient again within 30 days of discharge.

In [0]:
%sql
CREATE OR REPLACE TABLE health.gold.fact_encounter
COMMENT 'One row per encounter from FHIR and the HL7 feed combined. Pseudonymous keys only.'
AS
WITH base AS (
  SELECT
    sha2(concat(key_salt, '|encounter|', e.encounter_id), 256)  AS encounter_key,
    sha2(concat(key_salt, '|patient|', e.patient_id), 256)      AS patient_key,
    coalesce(o.organization_id, 'UNKNOWN')                       AS facility_key,
    e.class_code                                                 AS encounter_class,
    e.type_code,
    e.type_display,
    e.reason_code,
    e.start_ts                                                   AS admit_ts,
    e.end_ts                                                     AS discharge_ts,
    h.visit_number IS NOT NULL                                   AS in_hl7_feed,
    coalesce(h.had_transfer, false)                              AS had_transfer
  FROM health.silver.fhir_encounters e
  LEFT JOIN health.silver.fhir_organizations o ON o.organization_id = e.organization_id
  LEFT JOIN health.silver.encounter_xref x     ON x.encounter_id = e.encounter_id
  LEFT JOIN health.silver.hl7_encounters h     ON h.facility_code = x.facility_code AND h.visit_number = x.visit_number

  UNION ALL

  -- HL7 visits that could not be linked to FHIR (expected: none)
  SELECT
    sha2(concat(key_salt, '|hl7visit|', h.facility_code, '|', h.visit_number), 256),
    sha2(concat(key_salt, '|mrn|', h.mrn), 256),
    coalesce(o.organization_id, 'UNKNOWN'),
    CASE h.patient_class WHEN 'I' THEN 'IMP' WHEN 'E' THEN 'EMER' ELSE 'AMB' END,
    NULL, NULL, NULL,
    h.admit_ts, h.discharge_ts,
    true, h.had_transfer
  FROM health.silver.hl7_encounters h
  LEFT ANTI JOIN health.silver.encounter_xref x ON x.facility_code = h.facility_code AND x.visit_number = h.visit_number
  LEFT JOIN health.silver.fhir_organizations o ON upper(o.organization_name) = upper(h.facility_name)
),
next_inpatient AS (
  SELECT encounter_key,
         lead(admit_ts) OVER (PARTITION BY patient_key ORDER BY admit_ts) AS next_inpatient_admit_ts
  FROM base
  WHERE encounter_class = 'IMP'
)
SELECT
  b.encounter_key,
  b.patient_key,
  b.facility_key,
  CAST(date_format(b.admit_ts, 'yyyyMMdd') AS INT)     AS admit_date_key,
  CAST(date_format(b.discharge_ts, 'yyyyMMdd') AS INT) AS discharge_date_key,
  b.encounter_class,
  CASE b.encounter_class WHEN 'AMB' THEN 'Ambulatory' WHEN 'EMER' THEN 'Emergency' WHEN 'IMP' THEN 'Inpatient'
                         WHEN 'HH' THEN 'Home health' WHEN 'VR' THEN 'Virtual' ELSE b.encounter_class END AS encounter_class_name,
  b.type_code,
  b.type_display,
  b.reason_code,
  b.admit_ts,
  b.discharge_ts,
  round((unix_timestamp(b.discharge_ts) - unix_timestamp(b.admit_ts)) / 3600.0, 2) AS length_of_stay_hours,
  datediff(b.discharge_ts, b.admit_ts)                  AS length_of_stay_days,
  b.encounter_class = 'IMP'
    AND n.next_inpatient_admit_ts >= b.discharge_ts
    AND n.next_inpatient_admit_ts <= b.discharge_ts + INTERVAL 30 DAYS   AS readmitted_within_30d,
  b.in_hl7_feed,
  b.had_transfer,
  CASE WHEN b.in_hl7_feed THEN 'FHIR + HL7' ELSE 'FHIR' END AS source_systems
FROM base b
LEFT JOIN next_inpatient n ON n.encounter_key = b.encounter_key;

## fact_encounter_diagnosis (bridge)

In [0]:
%sql
CREATE OR REPLACE TABLE health.gold.fact_encounter_diagnosis
COMMENT 'Bridge between encounters and diagnoses: one row per encounter and SNOMED CT code.'
AS
SELECT DISTINCT
  sha2(concat(key_salt, '|encounter|', c.encounter_id), 256) AS encounter_key,
  c.code                                                      AS diagnosis_code
FROM health.silver.fhir_conditions c
WHERE c.encounter_id <> '' AND c.code IS NOT NULL;

## Validate

In [0]:
%sql
-- Grain check: exactly one row per encounter (both numbers must be equal, 5,087)
SELECT COUNT(*) AS fact_rows, COUNT(DISTINCT encounter_key) AS distinct_encounters
FROM health.gold.fact_encounter

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM health.gold.dim_patient)                                 AS patients,              -- expect 112
  (SELECT COUNT(*) FROM health.gold.dim_patient WHERE in_hl7_feed)               AS patients_in_hl7_feed,  -- expect 88
  (SELECT COUNT(*) FROM health.gold.dim_facility)                                AS facilities,            -- expect 268 (267 + Unknown)
  (SELECT COUNT(*) FROM health.gold.dim_diagnosis)                               AS diagnoses,             -- expect 190
  (SELECT COUNT(*) FROM health.gold.fact_encounter)                              AS encounters,            -- expect 5,087
  (SELECT COUNT(*) FROM health.gold.fact_encounter WHERE in_hl7_feed)            AS encounters_both_sources, -- expect 1,799
  (SELECT COUNT(*) FROM health.gold.fact_encounter f
     LEFT ANTI JOIN health.gold.dim_patient p USING (patient_key))               AS orphan_patients,       -- expect 0
  (SELECT COUNT(*) FROM health.gold.fact_encounter f
     LEFT ANTI JOIN health.gold.dim_facility d USING (facility_key))             AS orphan_facilities      -- expect 0

In [0]:
%sql
-- Inpatient headline numbers. Expect 94 stays, about 6 days average, 2 readmissions within 30 days.
SELECT
  COUNT(*)                                          AS inpatient_stays,
  round(avg(length_of_stay_hours) / 24, 2)          AS avg_length_of_stay_days,
  SUM(CASE WHEN readmitted_within_30d THEN 1 ELSE 0 END) AS readmissions_30d,
  SUM(CASE WHEN had_transfer THEN 1 ELSE 0 END)     AS stays_with_transfer
FROM health.gold.fact_encounter
WHERE encounter_class = 'IMP'

In [0]:
%sql
-- De-identification check: no direct identifiers in any gold column name
SELECT table_name, column_name
FROM health.information_schema.columns
WHERE table_schema = 'gold'
  AND lower(column_name) RLIKE 'name|birth_date|ssn|mrn|address|phone|patient_id'
  AND column_name NOT IN ('facility_name', 'diagnosis_name', 'month_name', 'day_name', 'encounter_class_name')
-- expect no rows